# Figure 7 — Archetypes of dynamics (S15, final)

S15 is complete (`RESULT_S15.md` status: all 130 analysis checkpoints present; `pending_units.csv` is a zero-row
completion record) and merged into `benchmarks/synthetic_multimodal/s15/` (`S15_DIR` in `_style.py`, env `SCJDO2_S15_DIR`).
Sources: `regime_recovery.csv` (A1), the gate block of `checkpoints/finalise.json` (A2; NMI per gate comparison is only
recorded there), `stability.csv` (A3); each plotted value is cross-checked against the gate block, and the gate
PASS/FAIL lines are cross-checked against `RESULT_S15.md`.

In [1]:
from _style import *
import json as _json, re

S15 = S15_DIR   # benchmarks/synthetic_multimodal/s15 (merged)
prov = Provenance("Fig7", notebook="notebooks/fig7_archetypes.ipynb")
rr = pd.read_csv(prov.use(S15 / "regime_recovery.csv", "a", "S15", note="final S15"))
stab = pd.read_csv(prov.use(S15 / "stability.csv", "c", "S15", note="final S15"))
md = prov.use(S15 / "RESULT_S15.md", "a,b,c", "S15", note="final S15; status and gate PASS/FAIL lines").read_text()
fin = _json.loads(prov.use(S15 / "checkpoints/finalise.json", "a,b,c", "S15", note="final S15; gate blocks").read_text())
assert "Complete:" in md.split("## Status", 1)[1].split("##", 1)[0], "S15 not complete"
assert len(pd.read_csv(prov.use(S15 / "pending_units.csv", "a,b,c", "S15", note="zero-row completion record"))) == 0
V = prov.value

def gate_json(name):
    g = fin["gates"][name]
    word = re.search(rf"\*\*{name}\*\*: (PASS|FAIL)", md).group(1)
    assert (word == "PASS") == g["pass"], name
    return g["by_view"]

gA1, gA2, gA3 = gate_json("gate_A1_saddle_archetype"), gate_json("gate_A2_nmi"), gate_json("gate_A3_stability")
for name, g in [("A1", gA1), ("A2", gA2), ("A3", gA3)]:
    V(f"gate_{name}_pass", {view: g[view]["pass"] for view in ("operator_archetype", "descriptor_kmeans")})
tests = ["v3_transition_s2", "v3_transition_s3", "v3_transition_s4"]
views = [("operator_archetype", "operator archetype\n(semi-NMF)"), ("descriptor_kmeans", "descriptor K-means")]

fig = plt.figure(figsize=(DOUBLE_COL, 2.7))
V("overall_pass", fin["gates"]["overall_pass"])
gs = fig.add_gridspec(1, 4, width_ratios=[1, 1, 1.25, 1.0], wspace=0.75)

In [2]:
ax_r = fig.add_subplot(gs[0, 0]); ax_p = fig.add_subplot(gs[0, 1])
sad = rr[(rr.regime == "saddle") & (rr.operator == "flow") & (rr.k == 15) & rr.dataset.isin(tests)]
for j, (view, lab) in enumerate(views):
    col = C_FLOW if view == "operator_archetype" else OKABE["skyblue"]
    g0 = sad[(sad.view == view) & (sad.drift_seed == 0)].set_index("dataset").loc[tests]
    # cross-check with the gate JSON
    js = {d["dataset"]: d for d in gA1[view]["saddle"]}
    for ds in tests:
        assert abs(js[ds]["recall"] - g0.loc[ds, "recall"]) < 1e-12
        assert abs(js[ds]["precision_over_base_rate"] - g0.loc[ds, "precision_over_base_rate"]) < 1e-12
    x = np.arange(3) + (j - 0.5) * 0.3
    ax_r.bar(x, g0.recall, width=0.28, color=col, label=lab.replace("\n", " "))
    ax_p.bar(x, g0.precision_over_base_rate, width=0.28, color=col)
    other = sad[(sad.view == view) & (sad.drift_seed != 0)]
    for i, ds in enumerate(tests):
        o = other[other.dataset == ds]
        ax_r.scatter([x[i]] * len(o), o.recall, s=5, color="black", zorder=3, lw=0)
        ax_p.scatter([x[i]] * len(o), o.precision_over_base_rate, s=5, color="black", zorder=3, lw=0)
    V(f"a_{view}_recall_min", g0.recall.min()); V(f"a_{view}_recall_max", g0.recall.max())
    V(f"a_{view}_prec_over_base_min", g0.precision_over_base_rate.min()); V(f"a_{view}_prec_over_base_max", g0.precision_over_base_rate.max())
    V(f"a_{view}_n_other_seed_rows", int(len(other)))
V("a_dk_n_seeds_below", int(sum(d["precision_over_base_rate"] < 3 for d in gA1["descriptor_kmeans"]["saddle"])))
ax_r.axhline(0.6, color="black", ls="--", lw=0.7); ax_p.axhline(3, color="black", ls="--", lw=0.7)
for ax in (ax_r, ax_p):
    ax.set_xticks(range(3)); ax.set_xticklabels([t.replace("v3_transition_", "") for t in tests]); ax.set_xlabel("test seed")
ax_r.set_ylabel("saddle recall (best archetype)"); ax_r.set_ylim(0, 1.05)
ax_p.set_ylabel("precision / saddle base rate")
ax_r.legend(fontsize=6.5, loc="lower left", bbox_to_anchor=(0.0, 1.1), ncol=2)
ax_r.set_title("A1 recall\n(gate ≥ 0.6)", fontsize=6.5); ax_p.set_title("A1 precision\n(gate ≥ 3× base rate)", fontsize=6.5)
panel_letter(ax_r, "a", x=-0.4)
prov.panel("a", "A1: saddle recall and precision over base rate of the best flow archetype per test seed and view (bars: drift seed 0, gate; dots: drift seeds 1-2, report only)")

In [3]:
ax = fig.add_subplot(gs[0, 2])
mk = {"nmi_flow": ("flow archetypes", C_FLOW, "o"), "nmi_score": ("score archetypes", C_SCORE, "s"), "leiden_nmi": ("Leiden (matched K)", C_BASE, "^")}
for j, (view, lab) in enumerate(views):
    comps = {c["dataset"]: c for c in gA2[view]["comparisons"]}
    for key, (l, col, m_) in mk.items():
        vals = [comps[d][key] for d in tests]
        xs = np.arange(3) + (j - 0.5) * 0.36
        ax.scatter(xs, vals, color=col if view == "operator_archetype" else "white", edgecolor=col, marker=m_, s=14, lw=0.8, zorder=3,
                   label=l if j == 0 else None)
        V(f"b_{view}_{key}_min", min(vals)); V(f"b_{view}_{key}_max", max(vals))
    n_beat_leiden = sum(comps[d]["nmi_flow"] > comps[d]["leiden_nmi"] for d in tests)
    n_beat_score = sum(comps[d]["nmi_flow"] > comps[d]["nmi_score"] for d in tests)
    V(f"b_{view}_n_seeds_flow_beats_leiden", int(n_beat_leiden)); V(f"b_{view}_n_seeds_flow_beats_score", int(n_beat_score))
    V(f"b_{view}_k_selected", sorted(set(comps[d]["k_selected"] for d in tests)))
ax.set_xticks(range(3)); ax.set_xticklabels([t.replace("v3_transition_", "") for t in tests]); ax.set_xlabel("test seed (left filled: operator view;\nright open: descriptor view)")
ax.set_ylabel("NMI with true regimes"); ax.set_ylim(0, 0.58); ax.set_yticks(np.arange(0, 0.41, 0.1))
ax.legend(fontsize=6, loc="upper left", borderaxespad=0.2, handletextpad=0.2)
ax.set_title("A2 regime\ninformation", fontsize=6.5)
panel_letter(ax, "b", x=-0.3)
prov.panel("b", "A2: NMI of flow archetypes, score archetypes and matched-K Leiden with the true regimes per test seed, both views")

In [4]:
ax = fig.add_subplot(gs[0, 3])
st = stab[(stab.comparison == "drift_seed") & (stab.k == 15) & stab.dataset.isin(tests)]
for j, (view, lab) in enumerate(views):
    col = C_FLOW if view == "operator_archetype" else OKABE["skyblue"]
    s = st[st.view == view]
    js = {(d["dataset"], d["left"], d["right"]): d["ari"] for d in gA3[view]["drift_seed_ari"]}
    for _, r in s.iterrows():
        assert abs(js[(r.dataset, r.left, r.right)] - r.ari) < 1e-12
    for i, ds in enumerate(tests):
        v = s[s.dataset == ds].ari
        ax.scatter(i + (j - 0.5) * 0.3 + np.array([-0.05, 0, 0.05])[:len(v)], v, color=col, s=12, zorder=3,
                   label=lab.replace("\n", " ") if i == 0 else None)
    V(f"c_{view}_ari_min", s.ari.min()); V(f"c_{view}_ari_max", s.ari.max()); V(f"c_{view}_n_pairs", int(len(s)))
ax.axhline(0.5, color="black", ls="--", lw=0.7)
ax.set_xticks(range(3)); ax.set_xticklabels([t.replace("v3_transition_", "") for t in tests]); ax.set_xlabel("test seed")
ax.set_ylabel("ARI between drift-fit seeds"); ax.set_ylim(-0.1, 1.0)
ax.set_title("A3 stability\n(gate ≥ 0.5)", fontsize=6.5)
panel_letter(ax, "c", x=-0.35)
prov.panel("c", "A3: assignment ARI between drift-fit seed pairs (0-1, 0-2, 1-2) per test seed and view, k=15")

save(fig, "Fig7")

[save] main/Fig7: 156.5 x 77.0 mm, PNG 3698 x 1818 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig7.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig7.png'))

In [5]:
v = prov.values
oa, dk = "operator_archetype", "descriptor_kmeans"
R = lambda a, b, nd=2: rng_str(v[a], v[b], nd)
PF = lambda b: "passes" if b else "fails"
prov.legend(f"""
**Figure 7 | Archetypes of dynamics (S15).**
Archetypes were fitted to unit-norm flow operators (semi-NMF, K by held-out reconstruction; operator-archetype view, K = {', '.join(map(str, v[f'b_{oa}_k_selected']))}) and by
K-means on rank-transformed basis-invariant descriptors (descriptor view, K = {', '.join(map(str, v[f'b_{dk}_k_selected']))}), on the three held-out v3_transition
test seeds (s2–s4; PCA-15; S14 reference flow J), and compared with score-operator archetypes and Leiden clustering of expression at matched K.
**a**, A1: recall of true saddle cells by the best-matching archetype and its precision relative to the saddle base rate (bars: drift seed 0, the
gate rows; dots: drift seeds 1–2, report only). Operator view: recall {R(f'a_{oa}_recall_min', f'a_{oa}_recall_max')}, precision
{rng_str(v[f'a_{oa}_prec_over_base_min'], v[f'a_{oa}_prec_over_base_max'], 1)}× base rate (A1 {PF(v['gate_A1_pass'][oa])}); descriptor view: recall {R(f'a_{dk}_recall_min', f'a_{dk}_recall_max')},
precision {rng_str(v[f'a_{dk}_prec_over_base_min'], v[f'a_{dk}_prec_over_base_max'], 1)}× (A1 {PF(v['gate_A1_pass'][dk])}; {v['a_dk_n_seeds_below']} of 3 seeds below 3×). Dashed lines: gate thresholds (recall ≥ 0.6, precision ≥ 3× base rate).
**b**, A2: normalised mutual information with all true regimes. Operator view: flow {R(f'b_{oa}_nmi_flow_min', f'b_{oa}_nmi_flow_max')}, Leiden
{R(f'b_{oa}_leiden_nmi_min', f'b_{oa}_leiden_nmi_max')}, score {R(f'b_{oa}_nmi_score_min', f'b_{oa}_nmi_score_max')}; flow beats score on
{v[f'b_{oa}_n_seeds_flow_beats_score']} of 3 seeds and Leiden on {v[f'b_{oa}_n_seeds_flow_beats_leiden']} of 3 (A2 {PF(v['gate_A2_pass'][oa])}). Descriptor view: flow
{R(f'b_{dk}_nmi_flow_min', f'b_{dk}_nmi_flow_max')}, Leiden {R(f'b_{dk}_leiden_nmi_min', f'b_{dk}_leiden_nmi_max', 3)}, score {R(f'b_{dk}_nmi_score_min', f'b_{dk}_nmi_score_max')}.
**c**, A3: adjusted Rand index of archetype assignments between drift-fit seeds 0, 1, 2 ({v[f'c_{oa}_n_pairs']} pairs per view): operator view
{R(f'c_{oa}_ari_min', f'c_{oa}_ari_max')}, descriptor view {R(f'c_{dk}_ari_min', f'c_{dk}_ari_max')} (gate ≥ 0.5 for every pair; operator view {PF(v['gate_A3_pass'][oa])}, descriptor view {PF(v['gate_A3_pass'][dk])}). Overall: {'one view satisfies' if v['overall_pass'] else 'no view satisfies'}
A1–A3 together (S15 complete).
""")
prov.write()